# Methods 5 – Transformer Encoder

## Objective
In this assignment, I'm building a neural network that can identify positive and negative movie reviews.
I'll use the IMDB dataset and add parts of a Transformer encoder. In the end, I'll test the model to see how well it works.

In [1]:

import keras
from keras.datasets import imdb

# Dataset settings
max_features = 10000
max_len = 250

# Load IMDB movie reviews
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=max_features
)

# Make all reviews the same length
x_train = keras.utils.pad_sequences(
    x_train, maxlen=max_len, padding="post"
)

x_test = keras.utils.pad_sequences(
    x_test, maxlen=max_len, padding="post"
)

print("Training data:", x_train.shape)
print("Test data:", x_test.shape)
print("First five labels:", y_train[:5])


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training data: (25000, 250)
Test data: (25000, 250)
First five labels: [1 0 0 1 0]


## Token and positional embedding
In this step, I turn token IDs into token vectors. I also create a vector for each position so the model can take word order into account. Finally, I add the token and position vectors together.

In [2]:

from keras.layers import Layer, Embedding

class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super().__init__()

        self.token_emb = Embedding(
            input_dim=vocab_size,
            output_dim=emb_dim
        )

        self.pos_emb = Embedding(
            input_dim=seq_len,
            output_dim=emb_dim
        )

    def call(self, x_input):
        seq_len = keras.ops.shape(x_input)[-1]

        positions = keras.ops.arange(
            start=0, stop=seq_len, step=1
        )

        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)

        return x_input + positions


# Test the embedding layer
embedding_layer = TokenAndPositionEmbedding(
    max_len, max_features, 32
)

sample = embedding_layer(x_train[:2])
print("Embedding shape:", sample.shape)


Embedding shape: (2, 250, 32)


## Multi-head attention and residual connection
I use multi-head self-attention to help the model learn how words relate to each other. The residual connection adds the original embeddings to the attention output so the original information can pass through. Then I normalize the feature values for each token to help stabilize training.

In [3]:
from keras.layers import Input, MultiHeadAttention, Add, LayerNormalization

embed_dim = 32
num_heads = 2

# Input and embeddings
inputs = Input(shape=(max_len,), dtype="int32")
x = TokenAndPositionEmbedding(
    max_len, max_features, embed_dim
)(inputs)

# Self-attention
attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim // num_heads
)(x, x)

# First residual connection
x = Add()([x, attention_output])

# First normalization
x = LayerNormalization(epsilon=1e-6)(x)

print("Shape after attention:", x.shape)



Shape after attention: (None, 250, 32)


## Feedforward network and second residual connection
Now I will use a feedforward network that uses two Dense layers for data management. In the first Dense layer with ReLU, there are 64 units, and the second one is linear. There are 32 units. The second layer will return it to 32 units. Then the second residual connection will connect the output and the feedforward result. The last thing to do is to use a layer normalization layer.

In [4]:

from keras.layers import Dense

# Feedforward network
ffn_output = Dense(64, activation="relu")(x)
ffn_output = Dense(embed_dim)(ffn_output)

# Second residual connection
x = Add()([x, ffn_output])

# Second normalization
x = LayerNormalization(epsilon=1e-6)(x)

print("Shape after feedforward:", x.shape)


Shape after feedforward: (None, 250, 32)


## Classification output
This is an important step. Here I will change the Transformer encoder's results into one prediction, and I will use a sigmoid function because the assignment involves two classes.

In [5]:

from keras.layers import GlobalAveragePooling1D, Dropout
from keras.models import Model

# Pooling and classification
x = GlobalAveragePooling1D()(x)
x = Dropout(0.5)(x)

outputs = Dense(1, activation="sigmoid")(x)

# Build the model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 250, 32)   │    328,000 │ input_layer[0][0] │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 32)   │      4,224 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 250, 32)   │          0 │ token_and_positi… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 250, 32)   │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 250, 64)   │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 250, 32)   │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 250, 32)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 32)   │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 336,577 (1.28 MB)

 Trainable params: 336,577 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

## Model training
Now it's time for the training step. I will be training the model using the IMDB training data, and meanwhile I will track the model's accuracy. I setted aside 20% of the training data for validation. 

In [6]:

# Train the model
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.7739 - loss: 0.4509 - val_accuracy: 0.8798 - val_loss: 0.2840
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9101 - loss: 0.2310 - val_accuracy: 0.8760 - val_loss: 0.2885
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9420 - loss: 0.1642 - val_accuracy: 0.8716 - val_loss: 0.3365
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9534 - loss: 0.1338 - val_accuracy: 0.8656 - val_loss: 0.3884
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.9596 - loss: 0.1135 - val_accuracy: 0.8552 - val_loss: 0.4651


## Model evaluation
Now I will test the trained model with different dataset, to see how good it can predict the positives and negatives reviews. 

In [7]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")


782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8416 - loss: 0.4939
Test accuracy: 0.8416
Test loss: 0.4939


## Results and observations
